In [ ]:
import pandas as pd
import yaml
import re
import os
DB_FOLDER = f"dataset\database_v3"
GRAPH_FOLDER = os.path.join(DB_FOLDER, "Graph")
os.makedirs(GRAPH_FOLDER, exist_ok=True)

In [2]:
trans_df = pd.read_csv(os.path.join(DB_FOLDER, "transaction.csv"))
trans_df

,transaction_id,timestep,Lease Commencement Date,project_id,Project Name,No of Bedroom,Floor Area (SQFT),sqft,Monthly Rent ($),rent_per_sqft
0,1,0,1999-11-01,1139,MANDARIN GARDENS,NaN,700 to 800,750.0,2015,2.686667
1,2,0,1999-11-01,1139,MANDARIN GARDENS,NaN,700 to 800,750.0,2325,3.100000
2,3,0,1999-11-01,1528,REGENT PARK,NaN,"1,100 to 1,200",1150.0,2500,2.173913
3,4,0,1999-11-01,276,CANNE LODGE,NaN,800 to 900,850.0,1700,2.000000
4,5,0,1999-11-01,276,CANNE LODGE,NaN,800 to 900,850.0,1500,1.764706
...,...,...,...,...,...,...,...,...,...,...
1260790,1260791,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,1.0,500 to 600,550.0,3600,6.545455
1260791,1260792,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,1.0,400 to 500,450.0,3500,7.777778
1260792,1260793,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,2.0,600 to 700,650.0,5200,8.000000
1260793,1260794,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,1.0,400 to 500,450.0,3700,8.222222


In [3]:
# Fill No of Bedroom
trans_df['No of Bedroom'] = trans_df['No of Bedroom'].fillna(0)
majority_map = (
    trans_df.loc[trans_df['No of Bedroom'] != 0]
    .groupby(['project_id', 'sqft'])['No of Bedroom']
    .agg(lambda x: x.mode().iloc[0])
)

mask = trans_df['No of Bedroom'] == 0

trans_df.loc[mask, 'No of Bedroom'] = trans_df.loc[mask, ['project_id', 'sqft']] \
    .apply(lambda row: majority_map.get((row['project_id'], row['sqft']), 0), axis=1)

In [4]:
trans_df

,transaction_id,timestep,Lease Commencement Date,project_id,Project Name,No of Bedroom,Floor Area (SQFT),sqft,Monthly Rent ($),rent_per_sqft
0,1,0,1999-11-01,1139,MANDARIN GARDENS,1.0,700 to 800,750.0,2015,2.686667
1,2,0,1999-11-01,1139,MANDARIN GARDENS,1.0,700 to 800,750.0,2325,3.100000
2,3,0,1999-11-01,1528,REGENT PARK,3.0,"1,100 to 1,200",1150.0,2500,2.173913
3,4,0,1999-11-01,276,CANNE LODGE,2.0,800 to 900,850.0,1700,2.000000
4,5,0,1999-11-01,276,CANNE LODGE,2.0,800 to 900,850.0,1500,1.764706
...,...,...,...,...,...,...,...,...,...,...
1260790,1260791,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,1.0,500 to 600,550.0,3600,6.545455
1260791,1260792,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,1.0,400 to 500,450.0,3500,7.777778
1260792,1260793,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,2.0,600 to 700,650.0,5200,8.000000
1260793,1260794,314,2026-01-01,691,FOURTH AVENUE RESIDENCES,1.0,400 to 500,450.0,3700,8.222222


In [5]:
grouped_df = (
    trans_df.groupby('project_id')
    .agg(
        count=('project_id', 'size')
    )
    .reset_index()
)
grouped_df = grouped_df.sort_values(['project_id']).reset_index(drop=True)
grouped_df['node_id'] = grouped_df.index
cols_order = ['node_id', 'project_id', "count"]
grouped_df = grouped_df[cols_order]
grouped_df.to_csv(os.path.join(GRAPH_FOLDER, "node_id.csv"), index=False)


In [6]:
grouped_df

,node_id,project_id,count
0,0,0,270
1,1,1,253
2,2,2,372
3,3,3,149
4,4,4,65
...,...,...,...
2492,2492,2492,57
2493,2493,2493,1
2494,2494,2494,782
2495,2495,2495,578


In [7]:
trans_df = (
    trans_df.groupby(['project_id', 'timestep'])
    .agg(
        lease_commencement_date=('Lease Commencement Date', 'first'),
        no_of_bedroom=('No of Bedroom', 'mean'),
        sqft=('sqft', 'mean'),
        rent_per_sqft=('rent_per_sqft', 'mean')
    )
    .reset_index()
)
trans_df.to_csv(os.path.join(GRAPH_FOLDER, "nodes.csv"), index=False)

In [8]:
trans_df

,project_id,timestep,lease_commencement_date,no_of_bedroom,sqft,rent_per_sqft
0,0,197,2016-04-01,1.615385,626.923077,3.887744
1,0,198,2016-05-01,1.200000,530.000000,4.044225
2,0,199,2016-06-01,1.400000,530.000000,4.265359
3,0,200,2016-07-01,1.000000,450.000000,4.333333
4,0,201,2016-08-01,1.000000,450.000000,4.111111
...,...,...,...,...,...,...
292280,2495,314,2026-01-01,1.500000,750.000000,5.958827
292281,2496,309,2025-08-01,2.500000,850.000000,4.065934
292282,2496,310,2025-09-01,2.000000,850.000000,5.884099
292283,2496,311,2025-10-01,2.333333,783.333333,5.577534


In [9]:
trans_df = trans_df.merge(
    grouped_df,
    on=['project_id'],  # important if grouped by timestep
    how='left'
)
trans_df = trans_df.drop(columns=['count'])
trans_df

,project_id,timestep,lease_commencement_date,no_of_bedroom,sqft,rent_per_sqft,node_id
0,0,197,2016-04-01,1.615385,626.923077,3.887744,0
1,0,198,2016-05-01,1.200000,530.000000,4.044225,0
2,0,199,2016-06-01,1.400000,530.000000,4.265359,0
3,0,200,2016-07-01,1.000000,450.000000,4.333333,0
4,0,201,2016-08-01,1.000000,450.000000,4.111111,0
...,...,...,...,...,...,...,...
292280,2495,314,2026-01-01,1.500000,750.000000,5.958827,2495
292281,2496,309,2025-08-01,2.500000,850.000000,4.065934,2496
292282,2496,310,2025-09-01,2.000000,850.000000,5.884099,2496
292283,2496,311,2025-10-01,2.333333,783.333333,5.577534,2496


In [10]:
# --------------------------------------------------
# Normalize trans_df features (same strategy as macro)
# --------------------------------------------------
# 1. Load existing config (if exists)
config_path = os.path.join(GRAPH_FOLDER, "config.yaml")
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
else:
    config = {}

    
feature_cols = ['no_of_bedroom', 'sqft']
SPLIT = 241


# 1. Train split (same SPLIT as macro)
train_mask = trans_df['timestep'] <= SPLIT
train_df = trans_df.loc[train_mask, feature_cols]

# 2. Compute scaler from TRAIN only
min_vals = train_df.min()
max_vals = train_df.max()
range_vals = (max_vals - min_vals).replace(0, 1)

# 3. Normalize ALL data (NaN stays NaN)
trans_df_norm = trans_df.copy()

trans_df_norm[feature_cols] = (
    2 * (trans_df[feature_cols] - min_vals) / range_vals - 1
)

# 4. Fill NaN AFTER normalization
trans_df_norm[feature_cols] = trans_df_norm[feature_cols].fillna(0)

trans_norm = {
    "method": "minmax_-1_1",
    "train_timestep_max": SPLIT,
    "features": {}
}

for col in feature_cols:
    trans_norm["features"][col] = {
        "min": float(min_vals[col]),
        "max": float(max_vals[col])
    }

config["trans_normalization"] = trans_norm

# save config
with open(config_path, "w") as f:
    yaml.dump(config, f)

In [11]:
import os
import pandas as pd

output_dir = os.path.join(GRAPH_FOLDER, "timestep")
os.makedirs(output_dir, exist_ok=True)

# ensure date is datetime
trans_df['lease_commencement_date'] = pd.to_datetime(
    trans_df['lease_commencement_date'], errors='coerce'
)

# columns to keep
cols = [
    'timestep', 'node_id', 'project_id',
    'no_of_bedroom', 'sqft',
    'rent_per_sqft', 'lease_commencement_date'
]

df_use = trans_df[cols].copy()

# --------------------------------------------------
# 1. aggregate observed rows first
# --------------------------------------------------
agg_df = (
    df_use.groupby(
        ['timestep', 'node_id', 'project_id'],
        as_index=False
    )
    .agg({
        'no_of_bedroom': 'mean',
        'sqft': 'mean',
        'rent_per_sqft': 'mean',
        'lease_commencement_date': 'first'
    })
)

agg_df['y_mask'] = 1

# --------------------------------------------------
# 2. interpolate missing timesteps within each node
# --------------------------------------------------
filled_list = []

for node_id, group in agg_df.groupby('node_id'):
    group = group.sort_values('timestep').copy()

    project_id = group['project_id'].iloc[0]

    full_timestep = pd.DataFrame({
        'timestep': range(
            int(group['timestep'].min()),
            int(group['timestep'].max()) + 1
        )
    })

    node_df = full_timestep.merge(
        group,
        on='timestep',
        how='left'
    )

    node_df['node_id'] = node_id
    node_df['project_id'] = node_df['project_id'].fillna(project_id)

    node_df['y_mask'] = node_df['y_mask'].fillna(0).astype(int)

    # interpolate numeric features
    node_df['no_of_bedroom'] = node_df['no_of_bedroom'].interpolate(method='linear')
    node_df['sqft'] = node_df['sqft'].interpolate(method='linear')
    node_df['rent_per_sqft'] = node_df['rent_per_sqft'].interpolate(method='linear')

    filled_list.append(node_df)

filled_df = pd.concat(filled_list, ignore_index=True)

# --------------------------------------------------
# 3. restore lease_commencement_date from timestep mapping
# --------------------------------------------------
timestep_date_map = (
    agg_df[['timestep', 'lease_commencement_date']]
    .drop_duplicates(subset=['timestep'])
    .sort_values('timestep')
)

filled_df = filled_df.drop(columns=['lease_commencement_date'], errors='ignore')

filled_df = filled_df.merge(
    timestep_date_map,
    on='timestep',
    how='left'
)

filled_df = filled_df.sort_values(['timestep', 'node_id']).reset_index(drop=True)

# --------------------------------------------------
# 4. split by timestep
# --------------------------------------------------
for t, group in filled_df.groupby('timestep'):
    group = group.sort_values('node_id')

    date = group['lease_commencement_date'].iloc[0]
    date_str = date.strftime('%Y-%m') if pd.notna(date) else 'unknown'

    file_name = f"{int(t)}_{date_str}.csv"
    file_path = os.path.join(output_dir, file_name)

    group_to_save = group[
        [
            'timestep',
            'node_id',
            'project_id',
            'no_of_bedroom',
            'sqft',
            'rent_per_sqft',
            'y_mask'
        ]
    ]

    group_to_save.to_csv(file_path, index=False)

In [12]:
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

timestep_dir = Path(GRAPH_FOLDER) / "timestep"

# --------------------------------------------------
# 1. Read feature tables
# --------------------------------------------------
project_df = pd.read_csv(os.path.join(DB_FOLDER, "project_processed.csv"))
school_df = pd.read_csv(os.path.join(DB_FOLDER, "school_processed.csv"))
mrt_df = pd.read_csv(os.path.join(DB_FOLDER, "mrt_processed.csv"))
proximity_df = pd.read_csv(os.path.join(DB_FOLDER, "proximity_processed.csv"))
amenities_df = pd.read_csv(os.path.join(DB_FOLDER, "amenities.csv"))
macro_df = pd.read_csv(os.path.join(DB_FOLDER, "macro_data_processed.csv"))

# --------------------------------------------------
# 2. Optional: remove duplicate rows by merge keys
# --------------------------------------------------
project_df = project_df.drop_duplicates(subset=["project_id"])
school_df = school_df.drop_duplicates(subset=["project_id"])
mrt_df = mrt_df.drop_duplicates(subset=["project_id"])
proximity_df = proximity_df.drop_duplicates(subset=["project_id"])
amenities_df = amenities_df.drop_duplicates(subset=["project_id"])
macro_df = macro_df.drop_duplicates(subset=["timestep"])

project_feature_dfs = [
    project_df,
    school_df,
    mrt_df,
    proximity_df,
    amenities_df
]

# --------------------------------------------------
# 3. Read each timestep CSV, merge features, and overwrite
# --------------------------------------------------
csv_files = sorted(timestep_dir.glob("*.csv"))
for csv_path in tqdm(csv_files, desc="Merging timestep files"):
    df = pd.read_csv(csv_path)

    # merge project-level features
    for feature_df in project_feature_dfs:
        df = df.merge(
            feature_df,
            on="project_id",
            how="left",
            suffixes=("", "_dup")
        )

        # remove duplicate columns created during merge
        dup_cols = [col for col in df.columns if col.endswith("_dup")]
        df = df.drop(columns=dup_cols)

    # merge macro-level features
    df = df.merge(
        macro_df,
        on="timestep",
        how="left",
        suffixes=("", "_dup")
    )

    dup_cols = [col for col in df.columns if col.endswith("_dup")]
    df = df.drop(columns=dup_cols)

    # --------------------------------------------------
    # 4. Ensure rent_per_sqft and y_mask are last columns
    # --------------------------------------------------
    last_cols = ["rent_per_sqft", "y_mask"]
    other_cols = [col for col in df.columns if col not in last_cols]

    df = df[other_cols + last_cols]

    # overwrite original csv
    df.to_csv(csv_path, index=False)

print("Finished merging features into all timestep CSV files.")

Merging timestep files: 100%|████████████████████████████████████████████████████████| 315/315 [01:09<00:00,  4.54it/s]

Finished merging features into all timestep CSV files.


In [16]:
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

timestep_dir = Path(GRAPH_FOLDER) / "timestep"
node_dir = Path(GRAPH_FOLDER) / "nodes"
node_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Read all timestep CSV files
# --------------------------------------------------
csv_files = sorted(timestep_dir.glob("*.csv"))

all_dfs = []

for csv_path in tqdm(csv_files, desc="Reading timestep files"):
    df = pd.read_csv(csv_path)
    all_dfs.append(df)

all_df = pd.concat(all_dfs, ignore_index=True)

# --------------------------------------------------
# 2. Sort by node_id and timestep
# --------------------------------------------------
all_df = all_df.sort_values(["node_id", "timestep"])

# --------------------------------------------------
# 3. Save one CSV per node (INCLUDING y_mask = 0)
# --------------------------------------------------
for node_id, node_df in tqdm(all_df.groupby("node_id"), desc="Saving node files"):
    node_df = node_df.sort_values("timestep").reset_index(drop=True)

    file_name = f"{int(node_id):04d}.csv"
    file_path = node_dir / file_name

    node_df.to_csv(file_path, index=False)

print(f"Finished saving node files to: {node_dir}")
print(f"Number of node files: {all_df['node_id'].nunique()}")

Saving node files: 100%|███████████████████████████████████████████████████████████| 2497/2497 [01:13<00:00, 33.93it/s]

Finished saving node files to: dataset\database_v3\Graph\nodes
Number of node files: 2497


## CCR Node ID Filtering

In [15]:
import os
import pandas as pd

ccr_districts = [1, 2, 9, 10, 11]

node_path = os.path.join(GRAPH_FOLDER, "node_id.csv")
project_path = os.path.join(DB_FOLDER, "project.csv")
output_path = os.path.join(GRAPH_FOLDER, "node_id_ccr.csv")

node_df = pd.read_csv(node_path)
project_df = pd.read_csv(project_path)

# keep only project_id + Postal District for mapping
project_map = project_df[["project_id", "Postal District"]].drop_duplicates("project_id")

# merge postal district into node_df
node_ccr_df = node_df.merge(
    project_map,
    on="project_id",
    how="left"
)

# filter CCR districts
node_ccr_df = node_ccr_df[
    node_ccr_df["Postal District"].isin(ccr_districts)
].copy()

# optionally drop Postal District if you only want original node_id.csv columns
node_ccr_df = node_ccr_df[node_df.columns]

node_ccr_df.to_csv(output_path, index=False)

print(f"Saved CCR node file to: {output_path}")
print(f"Number of CCR nodes: {len(node_ccr_df)}")

Saved CCR node file to: dataset\database_v3\Graph\node_id_ccr.csv
Number of CCR nodes: 803
